In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("IMDB Dataset.csv")

# Display first five rows
print(df.head())

# Check dataset shape
print("Dataset Shape:", df.shape)

# Check column names
print("Columns:", df.columns.tolist())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Dataset Shape: (50000, 2)
Columns: ['review', 'sentiment']


In [2]:
df.isnull().sum()

,0
review,0
sentiment,0


In [3]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 418


In [4]:
print(df["sentiment"].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [5]:

# Remove duplicate records
df = df.drop_duplicates().reset_index(drop=True)

# Check dataset after removing duplicates
print("Shape after removing duplicates:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

print("\nUpdated sentiment distribution:")
print(df["sentiment"].value_counts())


Shape after removing duplicates: (49582, 2)
Remaining duplicates: 0

Updated sentiment distribution:
sentiment
positive    24884
negative    24698
Name: count, dtype: int64


In [6]:

# Import regex library for text cleaning
import re

# Import NLTK for natural language processing
import nltk

# Import English stopwords list
from nltk.corpus import stopwords

# Import lemmatizer to convert words to their base form
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download("stopwords")  # For removing common English words
nltk.download("wordnet")    # For lemmatization
nltk.download("omw-1.4")    # Additional word-language data used by WordNet

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [7]:

# Initialize preprocessing tools
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    # 1. Convert text to lowercase
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # 3. Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # 4. Tokenize text into words
    words = text.split()

    # 5. Remove stopwords and apply lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    # 6. Join words back into a sentence
    return " ".join(words)


# Apply preprocessing to reviews
df["cleaned_review"] = df["review"].apply(preprocess_text)

# Display original and cleaned reviews
print(df[["review", "cleaned_review"]].head())


                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                      cleaned_review  
0  one reviewer mentioned watching oz episode hoo...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically family little boy jake think zombie ...  
4  petter mattei love time money visually stunnin...  


In [8]:
from sklearn.model_selection import train_test_split

# Split the dataset into 80% training and 20% testing data
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

# Display the shapes (rows and columns) of both datasets
print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

# Display sentiment counts in the training dataset
print("\nTraining sentiment distribution:")
print(train_df["sentiment"].value_counts())

# Display sentiment counts in the testing dataset
print("\nTesting sentiment distribution:")
print(test_df["sentiment"].value_counts())

Training data: (39665, 3)
Testing data: (9917, 3)

Training sentiment distribution:
sentiment
positive    19907
negative    19758
Name: count, dtype: int64

Testing sentiment distribution:
sentiment
positive    4977
negative    4940
Name: count, dtype: int64


In [9]:
# Import TF-IDF vectorizer to convert text into numerical features
from sklearn.feature_extraction.text import TfidfVectorizer

X_train = train_df["cleaned_review"]
X_test = test_df["cleaned_review"]

y_train = train_df["sentiment"]
y_test = test_df["sentiment"]


# Initialize TF-IDF vectorizer with a maximum of 5000 features
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))

Training TF-IDF shape: (39665, 5000)
Testing TF-IDF shape: (9917, 5000)
Number of features: 5000


In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Train Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

print("Multinomial Naive Bayes training complete!")
print("Logistic Regression training complete!")


Multinomial Naive Bayes training complete!
Logistic Regression training complete!


In [11]:
from sklearn.metrics import accuracy_score, classification_report

# Naive Bayes predictions
y_pred_nb = nb_model.predict(X_test_tfidf)

# Logistic Regression predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

print("=== Multinomial Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

=== Multinomial Naive Bayes ===
Accuracy: 0.8546939598668952
              precision    recall  f1-score   support

    negative       0.86      0.84      0.85      4940
    positive       0.85      0.86      0.86      4977

    accuracy                           0.85      9917
   macro avg       0.85      0.85      0.85      9917
weighted avg       0.85      0.85      0.85      9917

=== Logistic Regression ===
Accuracy: 0.889381869516991
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4940
    positive       0.88      0.90      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [12]:

def predict_sentiment(review):
    cleaned_review = preprocess_text(review)
    review_tfidf = tfidf.transform([cleaned_review])
    prediction = lr_model.predict(review_tfidf)

    return prediction[0]


# Test with custom reviews
print(predict_sentiment("This movie was amazing and wonderful!"))
print(predict_sentiment("This movie was boring and terrible."))


positive
negative
